# Virtual Embryo Challenge, Task 1: zero-data starter (SYNTHETIC data only)

Build and test a full T1 pipeline **before** you register or download anything:

1. generate a synthetic dataset in the documented T1 file layout (optionally on the real public
   32,285-gene panel, which is public metadata, not data);
2. run two baselines: `copy_last` (the official floor) and `composition_shift` (a simple
   composition-aware baseline);
3. validate each prediction against the documented submission contract, including the two silent
   killers: wrong normalisation and wrong gene order;
4. score with the official `veckit` metric functions and the documented 0-100 skill map.

**Nothing here is challenge data.** Synthetic numbers say nothing about the real board.
Runs as a script (`python t1_starter.py`) or cell by cell in Jupyter / Colab / VS Code (`# %%` cells).

In [ ]:
# Colab / fresh env only:
%pip install -q anndata scipy numpy pandas veckit==0.1.2

In [ ]:
from __future__ import annotations

import argparse
import json
import math
import sys
import tempfile
from dataclasses import dataclass, field
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

## 1. Synthetic T1-shaped data
Documented layout: AnnData `.h5ad`, `.X` float32 = log1p(1e4 * counts / total), var index = the board's
gene panel in its exact order, `obs["celltype"]` on training stages. Released T1 training stages are
E8.5 and E9.5; the validation target is E10.5. Our generative story mimics the documented difficulty
(cell-type composition turns over between stages; each type drifts), not the biology.

In [ ]:
STAGES = {"E8.5": 8.5, "E9.5": 9.5, "E10.5": 10.5}


def _type_table(rng, G, K):
    base = rng.normal(-1.0, 1.4, size=G)
    prof, drift, a, b = [], [], [], []
    for k in range(K):
        p = base.copy()
        prog = rng.choice(G, size=max(5, G // 25), replace=False)
        p[prog] += rng.normal(2.2, 0.6, size=prog.size)
        d = np.zeros(G)
        dg = rng.choice(G, size=max(5, G // 20), replace=False)
        d[dg] = rng.normal(0, 0.7, size=dg.size)
        prof.append(p); drift.append(d)
        if k < K // 3:
            a.append(rng.normal(1.0, 0.3)); b.append(rng.normal(-1.0, 0.2))     # declining
        elif k < 2 * K // 3:
            a.append(rng.normal(0.5, 0.3)); b.append(rng.normal(0.0, 0.15))     # stable
        else:
            a.append(rng.normal(-2.5, 0.4)); b.append(rng.normal(1.1, 0.2))     # emerging
    return np.array(prof), np.array(drift), np.array(a), np.array(b), [f"SynType{k:02d}" for k in range(K)]


def make_stage(rng, t, tbl, n_cells, genes):
    prof, drift, a, b, names = tbl
    logits = a + b * (t - 8.5)
    pk = np.exp(logits - logits.max()); pk /= pk.sum()
    ct = rng.choice(len(names), size=n_cells, p=pk)
    lam = prof[ct] + drift[ct] * (t - 8.5) + rng.normal(0, 0.35, size=(n_cells, len(genes)))
    lam = np.exp(lam - lam.max(1, keepdims=True)); lam /= lam.sum(1, keepdims=True)
    lib = np.exp(rng.normal(np.log(2500), 0.4, size=(n_cells, 1)))
    counts = rng.poisson(lam * lib).astype(np.float32)
    tot = counts.sum(1, keepdims=True); tot[tot == 0] = 1
    X = np.log1p(1e4 * counts / tot).astype(np.float32)
    obs = pd.DataFrame({"celltype": pd.Categorical([names[i] for i in ct])},
                       index=[f"SYN_{t}_{i}" for i in range(n_cells)])
    A = ad.AnnData(X=sparse.csr_matrix(X), obs=obs, var=pd.DataFrame(index=list(genes)))
    A.uns["SYNTHETIC"] = "generated by t1_starter.py; contains no challenge data"
    return A


def generate(n_genes=2000, n_types=9, cells=1500, seed=7, genes=None) -> dict:
    rng = np.random.default_rng(seed)
    genes = list(genes) if genes is not None else [f"Gsyn{i:05d}" for i in range(n_genes)]
    tbl = _type_table(rng, len(genes), n_types)
    return {s: make_stage(rng, t, tbl, cells, genes) for s, t in STAGES.items()}

## 2. Baselines
`copy_last` is the official floor ("Predict the last observed stage, verbatim"): it scores exactly 50.
`composition_shift` extrapolates cell-type proportions log-linearly in time, resamples last-stage cells to
them, and adds a damped per-type mean shift. A starting point, not a method.

In [ ]:
def _dense(X):
    return (X.toarray() if sparse.issparse(X) else np.asarray(X)).astype(np.float32, copy=False)


def copy_last(prev, last, **_):
    return last.copy()


def composition_shift(prev, last, *, t_prev, t_last, t_target, n_out=None, shift_damp=0.5, pseudo=2.0, seed=0):
    rng = np.random.default_rng(seed)
    Xp, Xl = _dense(prev.X), _dense(last.X)
    tp, tl = np.asarray(prev.obs["celltype"]).astype(str), np.asarray(last.obs["celltype"]).astype(str)
    types = np.unique(tl)
    n_out = n_out or last.n_obs
    dt, dT = t_last - t_prev, t_target - t_last
    cp = np.array([(tp == c).sum() for c in types], float) + pseudo
    cl = np.array([(tl == c).sum() for c in types], float) + pseudo
    lp, ll = np.log(cp / cp.sum()), np.log(cl / cl.sum())
    logit = ll + (ll - lp) / dt * dT
    p = np.exp(logit - logit.max()); p /= p.sum()
    rows = []
    for c, k in zip(types, rng.multinomial(n_out, p)):
        if k == 0:
            continue
        shift = Xl[tl == c].mean(0) - Xp[tp == c].mean(0) if (tp == c).any() else 0.0
        pick = rng.choice(np.flatnonzero(tl == c), size=k, replace=True)
        rows.append(Xl[pick] + shift_damp * (dT / dt) * shift)
    Y = np.maximum(np.vstack(rows), 0).astype(np.float32)
    return ad.AnnData(X=sparse.csr_matrix(Y), var=last.var.copy())

## 3. Validate against the documented contract
Mirrors the documented checks: exact gene order, finite, non-negative, cell-count limits. Plus a check the
official validator does NOT make but that silently ruins your score: the implied library size
(median of expm1(X).sum(1)) must be ~1e4, i.e. log1p(1e4-normalised), not raw counts or another target sum.
The portal's own format check is authoritative, free and unlimited (Rules s.11). Run it too.

In [ ]:
@dataclass
class Check:
    ok: bool = True
    errors: list = field(default_factory=list)
    warnings: list = field(default_factory=list)

    def err(self, m):
        self.ok = False
        self.errors.append(m)


def validate(a, genes, *, min_cells, max_cells) -> Check:
    c = Check()
    got = [str(g) for g in a.var_names]
    if got != list(genes):
        same_set = set(got) == set(genes)
        c.err("var_names: " + ("right genes, WRONG ORDER (rejected just the same)" if same_set else
                               f"expected {len(genes)} panel genes, got {len(got)}"))
    vals = a.X.data if sparse.issparse(a.X) else np.asarray(a.X)
    if not np.isfinite(vals).all():
        c.err(".X has NaN/inf")
    if (vals < 0).any():
        c.err(".X has negative values")
    if not (min_cells <= a.n_obs <= max_cells):
        c.err(f"n_obs={a.n_obs} outside [{min_cells}, {max_cells}]")
    d = _dense(a.X[: min(500, a.n_obs)])
    lib = float(np.median(np.expm1(np.clip(d, 0, 50)).sum(1)))
    if not (0.5e4 <= lib <= 2e4):
        c.warnings.append(f"implied library size {lib:.3g}, expected ~1e4: is .X log1p(1e4-normalised)? "
                          "This PASSES format checks but scores wrongly.")
    return c


def write_submission(a, path):
    out = ad.AnnData(X=sparse.csr_matrix(a.X, dtype=np.float32), var=pd.DataFrame(index=list(a.var_names)))
    out.obs_names = [f"pred_{i}" for i in range(out.n_obs)]
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    out.write_h5ad(path, compression="gzip")

## 4. Score: official veckit metrics + the documented skill map
Raw metrics come from veckit's own functions (`pip install veckit==0.1.2`, MIT). Anchors are computed on
the synthetic target exactly as documented: floor = copy_last, ceiling = one half of the target scored
against the other half. skill = min(d_floor / (d_floor + d), 1); T1 weights 25/25/30/20.
(For real boards use the published anchors in panels/index.json, e.g. with the vec-headline tool.)

In [ ]:
T1_WEIGHTS = {"de_score": 0.25, "de_direction": 0.25, "mmd_u": 0.30, "variogram": 0.20}
LOWER_IS_BETTER = {"de_score": False, "de_direction": False, "mmd_u": True, "variogram": True}


def t1_raw(pred_X, true_X, ref_X, seed=0):
    from common.core_metrics import de_direction, de_score, mmd_unbiased, variogram_score
    return {"de_score": de_score(pred_X, true_X, ref_X)["score"],
            "de_direction": de_direction(pred_X, true_X, ref_X),
            "mmd_u": mmd_unbiased(pred_X, true_X, seed=seed),
            "variogram": variogram_score(pred_X, true_X, seed=seed)}


def skill(v, f, c, lower):
    if not all(map(math.isfinite, (v, f, c))) or abs(c - f) < 1e-12:
        return 0.0
    d_floor, d = abs(c - f), (v - c) if lower else (c - v)
    return 1.0 if d_floor + d <= 1e-9 else min(d_floor / (d_floor + d), 1.0)


def headline(raw, anchors):
    sk = {m: skill(raw[m], anchors[m]["floor"], anchors[m]["ceiling"], LOWER_IS_BETTER[m]) for m in T1_WEIGHTS}
    return 100 * sum(T1_WEIGHTS[m] * sk[m] for m in T1_WEIGHTS), {m: 100 * s for m, s in sk.items()}


def synthetic_anchors(target, ref, seed=0):
    from common.core_metrics import split_half
    T, R = _dense(target.X), _dense(ref.X)
    a, b = split_half(T.shape[0], seed=seed)
    fl, ce = t1_raw(R, T[a], R, seed), t1_raw(T[b], T[a], R, seed)
    return T[a], {m: {"floor": fl[m], "ceiling": ce[m]} for m in fl}

## 5. Run it

In [ ]:
def run(out_dir, n_genes=2000, cells=1500, genes_file=None, seed=7, min_cells=None, max_cells=None, quiet=False):
    log = (lambda *a: None) if quiet else print
    out = Path(out_dir)
    genes = [g.strip() for g in Path(genes_file).read_text().splitlines() if g.strip()] if genes_file else None
    data = generate(n_genes=n_genes, cells=cells, seed=seed, genes=genes)
    genes = list(data["E8.5"].var_names)
    for s, A in data.items():
        sub = "released" if s != "E10.5" else "hidden"
        (out / sub).mkdir(parents=True, exist_ok=True)
        A.write_h5ad(out / sub / f"{s}_RNA.h5ad", compression="gzip")
    log(f"SYNTHETIC data: {len(genes)} genes x {cells} cells/stage -> {out}")
    # Official T1:val limits are 1000-5118 cells; scale down for small synthetic runs.
    lo = min_cells if min_cells is not None else min(1000, cells // 2)
    hi = max_cells if max_cells is not None else 5118
    prev, last, target = data["E8.5"], data["E9.5"], data["E10.5"]
    truth, anchors = synthetic_anchors(target, last)
    results = {}
    for name, fn in {"copy_last": copy_last, "composition_shift": composition_shift}.items():
        pred = fn(prev, last, t_prev=8.5, t_last=9.5, t_target=10.5)
        chk = validate(pred, genes, min_cells=lo, max_cells=hi)
        path = out / "preds" / f"{name}.h5ad"
        write_submission(pred, path)
        raw = t1_raw(_dense(pred.X), truth, _dense(last.X))
        h, sk = headline(raw, anchors)
        results[name] = {"valid": chk.ok, "errors": chk.errors, "warnings": chk.warnings,
                         "headline": h, "skill_x100": sk, "raw": raw, "file": str(path)}
    # The two silent killers, demonstrated on the composition_shift prediction:
    p = composition_shift(prev, last, t_prev=8.5, t_last=9.5, t_target=10.5)
    raw_counts = ad.AnnData(X=sparse.csr_matrix(np.expm1(_dense(p.X)) * 0.25), var=p.var.copy())
    shuffled = p[:, list(reversed(genes))].copy()
    results["_killers"] = {
        "raw_counts": vars(validate(raw_counts, genes, min_cells=lo, max_cells=hi)),
        "shuffled_genes": vars(validate(shuffled, genes, min_cells=lo, max_cells=hi)),
    }
    log("\nSYNTHETIC scores (copy_last floor = 50, ceiling = 100; NOT predictive of the real board)")
    log(f"  {'method':<18}{'valid':>6}{'headline':>10} | " + " ".join(f"{m:>12}" for m in T1_WEIGHTS))
    for n in ("copy_last", "composition_shift"):
        r = results[n]
        log(f"  {n:<18}{str(r['valid']):>6}{r['headline']:>10.2f} | "
            + " ".join(f"{r['skill_x100'][m]:>12.2f}" for m in T1_WEIGHTS))
    k = results["_killers"]
    log("\nSilent killers:")
    log(f"  raw counts     -> valid={k['raw_counts']['ok']} warnings={k['raw_counts']['warnings']}")
    log(f"  shuffled genes -> valid={k['shuffled_genes']['ok']} errors={k['shuffled_genes']['errors']}")
    (out / "results.json").write_text(json.dumps(results, indent=2, default=float))
    return results

In [ ]:
if __name__ == "__main__" and "ipykernel" not in sys.modules:
    ap = argparse.ArgumentParser(description="VEC T1 zero-data starter (synthetic only)")
    ap.add_argument("--out", default="synthetic_t1")
    ap.add_argument("--genes", type=int, default=2000)
    ap.add_argument("--cells", type=int, default=1500)
    ap.add_argument("--genes-file", help="public T1 panel (T1__val.genes.txt) for a full-width format test")
    ap.add_argument("--seed", type=int, default=7)
    a = ap.parse_args()
    run(a.out, n_genes=a.genes, cells=a.cells, genes_file=a.genes_file, seed=a.seed)
elif "ipykernel" in sys.modules:
    _results = run(tempfile.mkdtemp(prefix="vec_t1_"), n_genes=2000, cells=1500)